# Chapter 25
## Phase Response Curves (PRCs)
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter25.ipynb)

## About this chapter

A phase response curve (PRC) reports how much a single perturbation shifts a
neuron's firing phase: deliver the perturbation at phase $\varphi$ (fraction of
the cycle since the last spike) and measure the resulting phase shift $g(\varphi)$.
The theta neuron admits a closed form for its return map $f$ and PRC $g=f-\varphi$;
for conductance-based neurons (RTM, WB, HH, Erisir), $g$ is measured numerically
by splay-initializing a population across every phase and simulating each one
until it fires once. Perturbations here are either a synaptic pulse (a single
decaying conductance transient) or a direct voltage kick.

See [`README.md`](chapter25.md) for the full guide, including suggested order
and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit

## Theta Neuron: Return Map and PRC

Above threshold, the theta neuron (the smoothed QIF from Chapter 8) has a
closed-form phase map. `theta_f` gives the return map $f(\varphi)$: the phase
one cycle later, after a jump of size $dv$ added right after the previous
spike. `theta_prc` is the PRC itself, $g=f-\varphi$, and
`theta_prc_short_weak` is $\hat g=dg/d(\delta v)$, the linear-response
(infinitesimal-kick) limit of the PRC.

In [ ]:
def theta_f(tau_m=2., I=0.13, dv=0.1, N=100):
    phi = np.arange(N + 1) / N
    f = np.arctan(np.tan(np.pi * phi - np.pi / 2) + dv / np.sqrt(tau_m * I - 1 / 4)) / np.pi + 1 / 2
    return phi, f


def theta_prc(tau_m=2., I=0.13, dv=0.1, N=100):
    phi, f = theta_f(tau_m=tau_m, I=I, dv=dv, N=N)
    g = f - phi
    return phi, g


def theta_prc_short_weak(tau_m=2., I=0.13, N=200):
    phi = np.arange(N + 1) / N
    g_hat = 1 / np.pi / np.sqrt(tau_m * I - 1 / 4) / (1 + np.tan(np.pi * (phi - 1 / 2)) ** 2)
    return phi, g_hat


def plot_theta_f(phi, f):
    plt.figure(figsize=(5, 5))
    plt.plot(phi, f, '-k', linewidth=6)
    plt.plot([0, 1], [0, 1], '--k', linewidth=2)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel('$f$')
    plt.tight_layout()
    plt.show()


def plot_theta_prc(phi, g):
    plt.figure(figsize=(5, 5))
    plt.plot(phi, g, '-k', linewidth=6)
    plt.axis([0, 1, 0, 1])
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel('$g$')
    plt.tight_layout()
    plt.show()


def plot_theta_prc_short_weak(phi, g_hat):
    plt.figure(figsize=(5, 5))
    plt.plot(phi, g_hat, '-k', linewidth=6)
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel(r'$\hat{g}$')
    plt.tight_layout()
    plt.show()

In [ ]:
interact(lambda dv=0.1: plot_theta_f(*theta_f(dv=dv)), dv=(0.0, 0.3, 0.01));

In [ ]:
interact(lambda dv=0.1: plot_theta_prc(*theta_prc(dv=dv)), dv=(0.0, 0.3, 0.01));

In [ ]:
plot_theta_prc_short_weak(*theta_prc_short_weak())

## RTM Limit Cycle and Release Time Constant (shared by every RTM example below)

`rtm_init` finds the single-cell RTM limit cycle at a given drive (plain Heun
integration until the 5th spike) and interpolates $(v,h,n)$ at each requested
phase -- this is how every RTM PRC example below splay-initializes its
population of independent neurons, one per phase.

In [ ]:
@njit
def _rtm_alpha_h(v):
    return 0.128 * np.exp(-(v + 50.0) / 18.0)


@njit
def _rtm_alpha_m(v):
    return 0.32 * (v + 54.0) / (1.0 - np.exp(-(v + 54.0) / 4.0))


@njit
def _rtm_alpha_n(v):
    return 0.032 * (v + 52.0) / (1.0 - np.exp(-(v + 52.0) / 5.0))


@njit
def _rtm_beta_h(v):
    return 4.0 / (1.0 + np.exp(-(v + 27.0) / 5.0))


@njit
def _rtm_beta_m(v):
    return 0.28 * (v + 27.0) / (np.exp((v + 27.0) / 5.0) - 1.0)


@njit
def _rtm_beta_n(v):
    return 0.5 * np.exp(-(v + 57.0) / 40.0)


@njit
def _rtm_m_inf(v):
    am, bm = _rtm_alpha_m(v), _rtm_beta_m(v)
    return am / (am + bm)


@njit
def _rtm_h_inf(v):
    ah, bh = _rtm_alpha_h(v), _rtm_beta_h(v)
    return ah / (ah + bh)


@njit
def _rtm_n_inf(v):
    an, bn = _rtm_alpha_n(v), _rtm_beta_n(v)
    return an / (an + bn)


@njit
def tau_peak_function(tau_d, tau_r, tau_d_q):
    """Time (from a delta-function pulse of transmitter release) at which
    the synaptic gate s peaks."""
    dt = 0.01
    dt05 = dt / 2
    s, t = 0.0, 0.0
    s_inc = np.exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05 * s_inc
        s_inc_tmp = np.exp(-(t + dt05) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt * s_inc_tmp
        t = t + dt
        s_inc = np.exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


@njit
def tau_d_q_function(tau_d, tau_r, tau_hat):
    """Release time constant tau_d_q so that tau_peak_function reproduces
    the prescribed tau_hat (bisection, since there's no closed form)."""
    tau_d_q_left = 1.0
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


@njit
def rtm_init(i_ext, phi_vec, c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
             v_k=-100.0, v_na=50.0, v_l=-67.0, t_final=5000.0, dt=0.001, v0=-70.0):
    """Find the RTM limit cycle at i_ext (plain-float Heun integration
    until the 5th spike), then interpolate (v, h, n) at each phase in
    phi_vec (fraction of the last full period, measured from the 4th
    spike). Returns (len(phi_vec), 3) array of initial conditions and
    the period T (np.inf if i_ext is subthreshold)."""
    dt05 = dt / 2

    v = [v0]
    m = [_rtm_m_inf(v0)]
    h = [_rtm_h_inf(v0)]
    n = [_rtm_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 4 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk)
                 + g_l * (v_l - vk) + i_ext) / c
        h_inc = _rtm_alpha_h(vk) * (1 - hk) - _rtm_beta_h(vk) * hk
        n_inc = _rtm_alpha_n(vk) * (1 - nk) - _rtm_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(_rtm_m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = ((k) * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    if len(t_spikes) < 5:
        v_last, h_last, n_last = v[k], h[k], n[k]
        out = np.zeros((num, 3))
        for i in range(num):
            out[i, 0], out[i, 1], out[i, 2] = v_last, h_last, n_last
        return out, np.inf

    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T

## RTM PRC: Single Synaptic Pulse

Splay-initialize a population of independent RTM neurons across every phase,
then at time 0 deliver a single synaptic conductance pulse to all of them
(release variable $q(0)=1$, decaying with $\tau_{dq}$; the gate $s$ then
rises and decays with $\tau_r,\tau_d$). $g(\varphi)=1-\varphi-t_\ast/T$ is
the phase shift, where $t_\ast$ is each neuron's first post-pulse spike
time. `simulate_synaptic_pulse_prc` is shared with the interaction-function,
three-weak-pulses, and weak-pulse examples below, which just call it with
different `g_syn` (and sometimes `dt`).

In [ ]:
def simulate_synaptic_pulse_prc(g_syn=0.1, i_ext=0.30, N=200, dt=0.01,
                                 tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                                 c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2

    phi_vec = np.arange(1, N + 1) / N - 1 / (2 * N)
    initial_vector, T = rtm_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                  v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy()
    m = _rtm_m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    q = np.ones(N)
    s = np.zeros(N)

    t_star = np.full(N, np.nan)  # first-spike time of each (independent) neuron
    num_spikes = np.zeros(N, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                 - g_syn * s * v + i_ext) / c
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n
        q_inc = -q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) - g_syn * s_tmp * v_tmp + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _rtm_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T + 1 - phi_vec
    return phi_vec, g_vec, T


def plot_rtm_prc(phi_vec, g_vec):
    plt.figure(figsize=(5, 5))
    plt.plot(phi_vec, g_vec, '-k', linewidth=6)
    plt.plot([0, 1], [1, 0], '--k', linewidth=2)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\varphi$')
    plt.ylabel('$g$')
    plt.tight_layout()
    plt.show()

In [ ]:
interact(lambda g_syn=0.1: plot_rtm_prc(*simulate_synaptic_pulse_prc(g_syn=g_syn)[:2]),
          g_syn=(0.01, 0.3, 0.01));

## RTM Interaction Function

The interaction function $f(\varphi)=\varphi+g(\varphi)$ folds the PRC into
the phase-map form used in later chapters on synchronization: it is the same
`simulate_synaptic_pulse_prc` simulation as above, just plotted as $f$
instead of $g$.

In [ ]:
phi_vec, g_vec, T = simulate_synaptic_pulse_prc(g_syn=0.1)
f_vec = phi_vec + g_vec

plt.figure(figsize=(5, 5))
plt.plot(phi_vec, f_vec, '-k', linewidth=6)
plt.plot([0, 1], [0, 1], '--k', linewidth=2)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.gca().set_box_aspect(1)
plt.xlabel(r'$\varphi$')
plt.ylabel('$f$')
plt.tight_layout()
plt.show()

## Three Weak Synaptic Pulses

As $g_{\rm syn}$ shrinks, the PRC's shape stabilizes while its amplitude
shrinks roughly proportionally -- the linear-response regime. Halving
$g_{\rm syn}$ three times from $0.02$ shows the trend.

In [ ]:
def simulate_three_weak_prcs(g_syn_0=2e-2, **kwargs):
    g_syn_vec = [g_syn_0 / 2 ** ijk for ijk in (1, 2, 3)]
    results = [simulate_synaptic_pulse_prc(g_syn=g_syn, **kwargs) for g_syn in g_syn_vec]
    phi_vec = results[0][0]
    T = results[0][2]
    g_vec_list = [g_vec for _, g_vec, _ in results]
    return phi_vec, g_vec_list, g_syn_vec, T


phi_vec, g_vec_list, g_syn_vec, T = simulate_three_weak_prcs()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ijk, (ax, g_syn, g_vec) in enumerate(zip(axes, g_syn_vec, g_vec_list)):
    ax.plot(phi_vec, g_vec, '-k', linewidth=2)
    ax.axis([0, 1, 0, max(g_vec) * 2])
    ax.set_title(rf'$\overline{{g}}_{{\rm syn}}={g_syn:g}$')
    ax.set_yticks(np.array([0.04, 0.08]) / 2 ** ijk)
    if ijk == 0:
        ax.set_ylabel('$g$')
    ax.set_xlabel(r'$\varphi$')
plt.tight_layout()
plt.show()

## Weak Synaptic Pulse: Linear-Response Check

Two synaptic strengths a factor of 10 apart, both normalized by
$g_{\rm syn}$: if the PRC is in its linear-response regime, the normalized
curves should coincide.

In [ ]:
def simulate_weak_prc_comparison(g_syn_1=0.001, dt=0.001, **kwargs):
    g_syn_2 = g_syn_1 / 10
    phi_vec, g_vec, T = simulate_synaptic_pulse_prc(g_syn=g_syn_1, dt=dt, **kwargs)
    phi_vec_2, g_vec_2, T2 = simulate_synaptic_pulse_prc(g_syn=g_syn_2, dt=dt, **kwargs)
    return phi_vec, g_vec, g_syn_1, phi_vec_2, g_vec_2, g_syn_2, T2


phi_vec, g_vec, g_syn_1, phi_vec_2, g_vec_2, g_syn_2, T = simulate_weak_prc_comparison()

plt.figure(figsize=(5, 5))
plt.plot(phi_vec, g_vec / g_syn_1, '-k', linewidth=4)
plt.plot(phi_vec_2[9::20], g_vec_2[9::20] / g_syn_2, '.r', markersize=25)
plt.gca().set_box_aspect(1)
plt.xlabel(r'$\varphi$')
plt.ylabel(r'$g/\overline{g}_{\rm syn}$')
plt.tight_layout()
plt.show()

## RTM PRC: Instantaneous Voltage Kick

A different kind of perturbation: instead of a synaptic pulse, every
splay-initialized neuron simply gets $\Delta v$ added to its voltage at
time 0 (an idealized infinitely fast, infinitely brief input). `delta_v=4`
is a strong kick; `simulate_short_weak_prc_comparison` below reuses the same
function at much smaller (weak, linear-response) kicks.

In [ ]:
def simulate_voltage_kick_prc(delta_v=4.0, i_ext=0.30, N=200, dt=0.001,
                               c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    dt05 = dt / 2
    phi_vec = np.arange(1, N + 1) / N - 1 / (2 * N)
    initial_vector, T = rtm_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                  v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy() + delta_v
    m = _rtm_m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()

    t_star = np.full(N, np.nan)
    num_spikes = np.zeros(N, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
        n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v_old = v.copy()
        v = v + dt * v_inc
        m = _rtm_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T - phi_vec + 1
    return phi_vec, g_vec, T


interact(lambda delta_v=4.0: plot_rtm_prc(*simulate_voltage_kick_prc(delta_v=delta_v)[:2]),
         delta_v=(0.5, 10.0, 0.5));

## RTM PRC: Weak Voltage Kick, Linear-Response Check

Same voltage-kick perturbation, now at two small $\Delta v$ a factor of 10
apart; normalizing by $\Delta v$ again checks the linear-response regime.

In [ ]:
def simulate_short_weak_prc_comparison(delta_v_1=0.01, i_ext=0.30, **kwargs):
    delta_v_2 = delta_v_1 / 10
    phi_vec, g_vec, T = simulate_voltage_kick_prc(delta_v=delta_v_1, i_ext=i_ext, **kwargs)
    _, g_vec_2, _ = simulate_voltage_kick_prc(delta_v=delta_v_2, i_ext=i_ext, **kwargs)
    return phi_vec, g_vec, delta_v_1, g_vec_2, delta_v_2, T


phi_vec, g_vec, delta_v_1, g_vec_2, delta_v_2, T = simulate_short_weak_prc_comparison()

plt.figure(figsize=(5, 5))
plt.plot(phi_vec, g_vec / delta_v_1, '-k', linewidth=6)
plt.plot(phi_vec[9::20], g_vec_2[9::20] / delta_v_2, '.r', markersize=25)
plt.axis([0, 1, 0, 0.15])
plt.gca().set_box_aspect(1)
plt.xlabel(r'$\varphi$')
plt.ylabel(r'$g/\Delta v$')
plt.tight_layout()
plt.show()

## Phase Shift From a Single Pulse

Rather than reading off a PRC curve, watch the shift directly: one RTM
neuron runs freely (`simulate_baseline`) while an identical one gets a
single synaptic pulse at `t_pulse` (`simulate_perturbed`) -- the horizontal
gap between the two traces after the pulse is the phase shift in action.

In [ ]:
def simulate_baseline(v0, h0, n0, i_ext=0.3, t_final=300.0, dt=0.01,
                       c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    v[0], h[0], n[0] = v0, h0, n0
    m[0] = _rtm_m_inf(v0)

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k])
                 + g_l * (v_l - v[k]) + i_ext) / c
        n_inc = _rtm_alpha_n(v[k]) * (1 - n[k]) - _rtm_beta_n(v[k]) * n[k]
        h_inc = _rtm_alpha_h(v[k]) * (1 - h[k]) - _rtm_beta_h(v[k]) * h[k]

        v_tmp = v[k] + dt05 * v_inc
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        m_tmp = _rtm_m_inf(v_tmp)

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

        v[k + 1] = v[k] + dt * v_inc
        m[k + 1] = _rtm_m_inf(v[k + 1])
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc

    return v


def simulate_perturbed(v0, h0, n0, t_pulse, g_syn=0.1, tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                        i_ext=0.3, t_final=300.0, dt=0.01,
                        c=1.0, g_k=80.0, g_na=100.0, g_l=0.1, v_k=-100.0, v_na=50.0, v_l=-67.0):
    """RTM neuron with a single synaptic input pulse (q set to 1) delivered
    at time t_pulse."""
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    q = np.zeros(m_steps + 1)
    s = np.zeros(m_steps + 1)
    v[0], h[0], n[0] = v0, h0, n0
    m[0] = _rtm_m_inf(v0)

    for k in range(m_steps):
        v_inc = (g_k * n[k] ** 4 * (v_k - v[k]) + g_na * m[k] ** 3 * h[k] * (v_na - v[k])
                 + g_l * (v_l - v[k]) + g_syn * s[k] * (-v[k]) + i_ext) / c
        n_inc = _rtm_alpha_n(v[k]) * (1 - n[k]) - _rtm_beta_n(v[k]) * n[k]
        h_inc = _rtm_alpha_h(v[k]) * (1 - h[k]) - _rtm_beta_h(v[k]) * h[k]
        q_inc = -q[k] / tau_dq
        s_inc = q[k] * (1 - s[k]) / tau_r - s[k] / tau_d

        v_tmp = v[k] + dt05 * v_inc
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        m_tmp = _rtm_m_inf(v_tmp)
        q_tmp = q[k] + dt05 * q_inc
        s_tmp = s[k] + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_syn * s_tmp * (-v_tmp) + i_ext) / c
        h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
        n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v[k + 1] = v[k] + dt * v_inc
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        m[k + 1] = _rtm_m_inf(v[k + 1])
        q[k + 1] = q[k] + dt * q_inc
        s[k + 1] = s[k] + dt * s_inc

        if abs((k + 1) * dt - t_pulse) < 1e-6:
            q[k + 1] = 1.0

    return v


def simulate_phase_shift(t_pulse=120.0, phi0=0.2, i_ext=0.3, g_syn=0.1,
                          tau_r=0.5, tau_peak=0.5, tau_d=2.0, t_final=300.0, dt=0.01):
    v0, h0, n0 = rtm_init(i_ext, np.array([phi0]))[0][0]
    v_baseline = simulate_baseline(v0, h0, n0, i_ext=i_ext, t_final=t_final, dt=dt)
    v_perturbed = simulate_perturbed(v0, h0, n0, t_pulse, g_syn=g_syn, tau_r=tau_r, tau_peak=tau_peak,
                                      tau_d=tau_d, i_ext=i_ext, t_final=t_final, dt=dt)
    t = np.arange(round(t_final / dt) + 1) * dt
    return t, v_baseline, v_perturbed


def plot_phase_shift(t, v_baseline, v_perturbed, t_pulse):
    plt.figure(figsize=(8, 4))
    plt.plot(t, v_baseline, '-b', linewidth=4)
    plt.axvline(t_pulse, linestyle='--', color='k', linewidth=1)
    plt.plot(t, v_perturbed, '-r', linewidth=1)
    plt.xlabel('$t$ [ms]')
    plt.ylabel('$v$ [mV]')
    plt.tight_layout()
    plt.show()

In [ ]:
interact(lambda t_pulse=120.0: plot_phase_shift(*simulate_phase_shift(t_pulse=t_pulse), t_pulse),
          t_pulse=(50.0, 250.0, 5.0));

## WB PRC (shared by the inhibitory-pulse example and the WB panel below)

Same splay-initialize-and-pulse recipe as the RTM synaptic-pulse PRC above,
now for the WB neuron. `simulate_wb_prc` takes the reversal potential
`v_rev` as an argument, so it covers both an inhibitory pulse ($v_{\rm
rev}=-70$mV, `WB_PRC_INHIBITORY_PULSE`) and an excitatory one ($v_{\rm
rev}=0$, the WB panel of `MISC_PRC` below).

In [ ]:
@njit
def _wb_alpha_h(v):
    return 0.35 * np.exp(-(v + 58.0) / 20.0)


@njit
def _wb_alpha_m(v):
    return 0.1 * (v + 35.0) / (1.0 - np.exp(-(v + 35.0) / 10.0))


@njit
def _wb_alpha_n(v):
    return 0.05 * (v + 34.0) / (1.0 - np.exp(-0.1 * (v + 34.0)))


@njit
def _wb_beta_h(v):
    return 5.0 / (np.exp(-0.1 * (v + 28.0)) + 1.0)


@njit
def _wb_beta_m(v):
    return 4.0 * np.exp(-(v + 60.0) / 18.0)


@njit
def _wb_beta_n(v):
    return 0.625 * np.exp(-(v + 44.0) / 80.0)


@njit
def _wb_m_inf(v):
    am, bm = _wb_alpha_m(v), _wb_beta_m(v)
    return am / (am + bm)


@njit
def _wb_h_inf(v):
    ah, bh = _wb_alpha_h(v), _wb_beta_h(v)
    return ah / (ah + bh)


@njit
def _wb_n_inf(v):
    an, bn = _wb_alpha_n(v), _wb_beta_n(v)
    return an / (an + bn)


@njit
def wb_init(i_ext, phi_vec, c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
            v_k=-90.0, v_na=55.0, v_l=-65.0, t_final=5000.0, dt=0.01, v0=-70.0):
    dt05 = dt / 2

    v = [v0]
    m = [_wb_m_inf(v0)]
    h = [_wb_h_inf(v0)]
    n = [_wb_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 4 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk) + g_l * (v_l - vk) + i_ext) / c
        h_inc = _wb_alpha_h(vk) * (1 - hk) - _wb_beta_h(vk) * hk
        n_inc = _wb_alpha_n(vk) * (1 - nk) - _wb_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = _wb_m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _wb_alpha_h(v_tmp) * (1 - h_tmp) - _wb_beta_h(v_tmp) * h_tmp
        n_inc = _wb_alpha_n(v_tmp) * (1 - n_tmp) - _wb_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(_wb_m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = (k * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T


def simulate_wb_prc(i_ext=1.0, g_syn=0.5, v_rev=-70.0, N=500, dt=0.01,
                     tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                     c=1.0, g_k=9.0, g_na=35.0, g_l=0.1, v_k=-90.0, v_na=55.0, v_l=-65.0):
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2

    phi_vec = np.arange(1, N + 1) / N - 1 / (2 * N)
    initial_vector, T = wb_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                 v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy()
    m = _wb_m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    q = np.ones(N)
    s = np.zeros(N)

    t_star = np.full(N, np.nan)
    num_spikes = np.zeros(N, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                 + g_syn * s * (v_rev - v) + i_ext) / c
        h_inc = _wb_alpha_h(v) * (1 - h) - _wb_beta_h(v) * h
        n_inc = _wb_alpha_n(v) * (1 - n) - _wb_beta_n(v) * n
        q_inc = -q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _wb_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_syn * s_tmp * (v_rev - v_tmp) + i_ext) / c
        h_inc = _wb_alpha_h(v_tmp) * (1 - h_tmp) - _wb_beta_h(v_tmp) * h_tmp
        n_inc = _wb_alpha_n(v_tmp) * (1 - n_tmp) - _wb_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _wb_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T + 1 - phi_vec
    return phi_vec, g_vec, T

### WB PRC: Inhibitory Pulse

A brief GABAergic-like pulse ($v_{\rm rev}=-70$mV, well below rest) can
still *advance* part of the cycle -- released inhibition lets $I_h$/leak
currents repolarize faster than they otherwise would, an example of a
non-monotonic PRC.

In [ ]:
phi_vec, g_vec, T = simulate_wb_prc()

plt.figure(figsize=(5, 5))
plt.plot(phi_vec, g_vec, '-k', linewidth=4)
plt.axis([0, 1, -0.5, 0.5])
plt.gca().set_box_aspect(1)
plt.xlabel(r'$\varphi$')
plt.ylabel('$g$')
plt.tight_layout()
plt.show()

## PRC Shapes Across Model Types (WB, HH, Erisir)

The same excitatory-synaptic-pulse PRC recipe, applied to three different
spiking models side by side: WB (reused from above with $v_{\rm rev}=0$),
the classical HH equations, and the type-2 Erisir model. Comparing panels
shows how the PRC's sign and shape depend on excitability type rather than
being a universal curve.

In [ ]:
@njit
def _hh_alpha_h(v):
    return 0.07 * np.exp(-(v + 70.0) / 20.0)


@njit
def _hh_alpha_m(v):
    """Scalar form (for hh_init's step-by-step integration)."""
    if abs(v + 45.0) > 1e-8:
        return (v + 45.0) / 10.0 / (1.0 - np.exp(-(v + 45.0) / 10.0))
    return 1.0


def _hh_alpha_m_vec(v):
    """Vectorized form (for simulate_hh_prc's array-valued population loop)."""
    return np.where(np.abs(v + 45.0) > 1e-8, (v + 45.0) / 10.0 / (1.0 - np.exp(-(v + 45.0) / 10.0)), 1.0)


@njit
def _hh_alpha_n(v):
    return 0.01 * (-60.0 - v) / (np.exp((-60.0 - v) / 10.0) - 1.0)


@njit
def _hh_beta_h(v):
    return 1.0 / (np.exp(-(v + 40.0) / 10.0) + 1.0)


@njit
def _hh_beta_m(v):
    return 4.0 * np.exp(-(v + 70.0) / 18.0)


@njit
def _hh_beta_n(v):
    return 0.125 * np.exp(-(v + 70.0) / 80.0)


@njit
def _hh_m_inf(v):
    am, bm = _hh_alpha_m(v), _hh_beta_m(v)
    return am / (am + bm)


@njit
def _hh_h_inf(v):
    ah, bh = _hh_alpha_h(v), _hh_beta_h(v)
    return ah / (ah + bh)


@njit
def _hh_n_inf(v):
    an, bn = _hh_alpha_n(v), _hh_beta_n(v)
    return an / (an + bn)


@njit
def hh_init(i_ext, phi_vec, c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
            v_k=-82.0, v_na=45.0, v_l=-59.0, t_final=5000.0, dt=0.005, v0=-70.0):
    """Unlike the RTM/WB/Erisir _init helpers, m is tracked as an explicit
    dynamic variable here (matching the original HH port), not m_inf(v)."""
    dt05 = dt / 2

    v = [v0]
    m = [_hh_m_inf(v0)]
    h = [_hh_h_inf(v0)]
    n = [_hh_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 4 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk) + g_l * (v_l - vk) + i_ext) / c
        m_inc = _hh_alpha_m(vk) * (1 - mk) - _hh_beta_m(vk) * mk
        h_inc = _hh_alpha_h(vk) * (1 - hk) - _hh_beta_h(vk) * hk
        n_inc = _hh_alpha_n(vk) * (1 - nk) - _hh_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = mk + dt05 * m_inc
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        m_inc = _hh_alpha_m(v_tmp) * (1 - m_tmp) - _hh_beta_m(v_tmp) * m_tmp
        h_inc = _hh_alpha_h(v_tmp) * (1 - h_tmp) - _hh_beta_h(v_tmp) * h_tmp
        n_inc = _hh_alpha_n(v_tmp) * (1 - n_tmp) - _hh_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        m.append(mk + dt * m_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)

        if vk >= -20 and v[-1] < -20:
            t_spike = (k * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 4))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = m[kk + 1] * frac_hi + m[kk] * frac_lo
        out[i, 2] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 3] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T


def simulate_hh_prc(i_ext=10.0, g_syn=0.1, v_rev=0.0, N=500, dt=0.005,
                     tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                     c=1.0, g_k=36.0, g_na=120.0, g_l=0.3, v_k=-82.0, v_na=45.0, v_l=-59.0):
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2

    phi_vec = np.arange(1, N + 1) / N - 1 / (2 * N)
    initial_vector, T = hh_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                 v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy()
    m = initial_vector[:, 1].copy()
    h = initial_vector[:, 2].copy()
    n = initial_vector[:, 3].copy()
    q = np.ones(N)
    s = np.zeros(N)

    t_star = np.full(N, np.nan)
    num_spikes = np.zeros(N, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                 + g_syn * s * (v_rev - v) + i_ext) / c
        m_inc = _hh_alpha_m_vec(v) * (1 - m) - _hh_beta_m(v) * m
        h_inc = _hh_alpha_h(v) * (1 - h) - _hh_beta_h(v) * h
        n_inc = _hh_alpha_n(v) * (1 - n) - _hh_beta_n(v) * n
        q_inc = -q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = m + dt05 * m_inc
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_syn * s_tmp * (v_rev - v_tmp) + i_ext) / c
        m_inc = _hh_alpha_m_vec(v_tmp) * (1 - m_tmp) - _hh_beta_m(v_tmp) * m_tmp
        h_inc = _hh_alpha_h(v_tmp) * (1 - h_tmp) - _hh_beta_h(v_tmp) * h_tmp
        n_inc = _hh_alpha_n(v_tmp) * (1 - n_tmp) - _hh_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = m + dt * m_inc
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T + 1 - phi_vec
    return phi_vec, g_vec, T


@njit
def _er_alpha_h(v):
    return 0.0035 / np.exp(v / 24.186)


@njit
def _er_alpha_m(v):
    return 40.0 * (75.5 - v) / (np.exp((75.5 - v) / 13.5) - 1.0)


@njit
def _er_alpha_n(v):
    return (95.0 - v) / (np.exp((95.0 - v) / 11.8) - 1.0)


@njit
def _er_beta_h(v):
    return -0.017 * (v + 51.25) / (np.exp(-(v + 51.25) / 5.2) - 1.0)


@njit
def _er_beta_m(v):
    return 1.2262 / np.exp(v / 42.248)


@njit
def _er_beta_n(v):
    return 0.025 / np.exp(v / 22.222)


@njit
def _er_m_inf(v):
    am, bm = _er_alpha_m(v), _er_beta_m(v)
    return am / (am + bm)


@njit
def _er_h_inf(v):
    ah, bh = _er_alpha_h(v), _er_beta_h(v)
    return ah / (ah + bh)


@njit
def _er_n_inf(v):
    an, bn = _er_alpha_n(v), _er_beta_n(v)
    return an / (an + bn)


@njit
def erisir_init(i_ext, phi_vec, c=1.0, g_k=224.0, g_na=112.0, g_l=0.5,
                v_k=-90.0, v_na=60.0, v_l=-70.0, t_final=5000.0, dt=0.002, v0=-70.0):
    """Erisir's K+ current has power 2, not 4 (n ** 2 below)."""
    dt05 = dt / 2

    v = [v0]
    m = [_er_m_inf(v0)]
    h = [_er_h_inf(v0)]
    n = [_er_n_inf(v0)]
    t_spikes = []

    k = 0
    t = 0.0
    while len(t_spikes) < 5 and t < t_final:
        vk, mk, hk, nk = v[k], m[k], h[k], n[k]
        v_inc = (g_k * nk ** 2 * (v_k - vk) + g_na * mk ** 3 * hk * (v_na - vk) + g_l * (v_l - vk) + i_ext) / c
        h_inc = _er_alpha_h(vk) * (1 - hk) - _er_beta_h(vk) * hk
        n_inc = _er_alpha_n(vk) * (1 - nk) - _er_beta_n(vk) * nk

        v_tmp = vk + dt05 * v_inc
        m_tmp = _er_m_inf(v_tmp)
        h_tmp = hk + dt05 * h_inc
        n_tmp = nk + dt05 * n_inc

        v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = _er_alpha_h(v_tmp) * (1 - h_tmp) - _er_beta_h(v_tmp) * h_tmp
        n_inc = _er_alpha_n(v_tmp) * (1 - n_tmp) - _er_beta_n(v_tmp) * n_tmp

        v.append(vk + dt * v_inc)
        h.append(hk + dt * h_inc)
        n.append(nk + dt * n_inc)
        m.append(_er_m_inf(v[-1]))

        if vk >= -20 and v[-1] < -20:
            t_spike = (k * dt * (-20 - v[-1]) + (k + 1) * dt * (20 + vk)) / (vk - v[-1])
            t_spikes.append(t_spike)
        t = (k + 1) * dt
        k += 1

    num = len(phi_vec)
    T = t_spikes[4] - t_spikes[3]
    out = np.zeros((num, 3))
    for i, phi0 in enumerate(phi_vec):
        t0 = phi0 * T + t_spikes[3]
        kk = int(t0 / dt)
        frac_hi = (t0 - kk * dt) / dt
        frac_lo = ((kk + 1) * dt - t0) / dt
        out[i, 0] = v[kk + 1] * frac_hi + v[kk] * frac_lo
        out[i, 1] = h[kk + 1] * frac_hi + h[kk] * frac_lo
        out[i, 2] = n[kk + 1] * frac_hi + n[kk] * frac_lo
    return out, T


def simulate_erisir_prc(i_ext=7.1, g_syn=0.1, v_rev=0.0, N=200, dt=0.002,
                         tau_r=0.5, tau_peak=0.5, tau_d=2.0,
                         c=1.0, g_k=224.0, g_na=112.0, g_l=0.5, v_k=-90.0, v_na=60.0, v_l=-70.0):
    tau_dq = tau_d_q_function(tau_d, tau_r, tau_peak)
    dt05 = dt / 2

    phi_vec = np.arange(1, N + 1) / N - 1 / (2 * N)
    initial_vector, T = erisir_init(i_ext, phi_vec, c=c, g_k=g_k, g_na=g_na, g_l=g_l,
                                     v_k=v_k, v_na=v_na, v_l=v_l)

    v = initial_vector[:, 0].copy()
    m = _er_m_inf(v)
    h = initial_vector[:, 1].copy()
    n = initial_vector[:, 2].copy()
    q = np.ones(N)
    s = np.zeros(N)

    t_star = np.full(N, np.nan)
    num_spikes = np.zeros(N, dtype=int)

    k = 0
    while np.min(num_spikes) < 1:
        k += 1

        v_inc = (g_k * n ** 2 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v)
                 + g_syn * s * (v_rev - v) + i_ext) / c
        h_inc = _er_alpha_h(v) * (1 - h) - _er_beta_h(v) * h
        n_inc = _er_alpha_n(v) * (1 - n) - _er_beta_n(v) * n
        q_inc = -q / tau_dq
        s_inc = q * (1 - s) / tau_r - s / tau_d

        v_tmp = v + dt05 * v_inc
        m_tmp = _er_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        q_tmp = q + dt05 * q_inc
        s_tmp = s + dt05 * s_inc

        v_inc = (g_k * n_tmp ** 2 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + g_syn * s_tmp * (v_rev - v_tmp) + i_ext) / c
        h_inc = _er_alpha_h(v_tmp) * (1 - h_tmp) - _er_beta_h(v_tmp) * h_tmp
        n_inc = _er_alpha_n(v_tmp) * (1 - n_tmp) - _er_beta_n(v_tmp) * n_tmp
        q_inc = -q_tmp / tau_dq
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v_old = v.copy()
        v = v + dt * v_inc
        m = _er_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        q = q + dt * q_inc
        s = s + dt * s_inc

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for i in ind:
            if num_spikes[i] == 0:
                t_star[i] = ((k - 1) * dt * (-20 - v[i]) + k * dt * (v_old[i] + 20)) / (v_old[i] - v[i])
            num_spikes[i] += 1

    g_vec = -t_star / T + 1 - phi_vec
    return phi_vec, g_vec, T

In [ ]:
phi_vec_wb, g_vec_wb, T_wb = simulate_wb_prc(i_ext=0.30, g_syn=0.1, v_rev=0.0, N=200, dt=0.01)
phi_vec_hh, g_vec_hh, T_hh = simulate_hh_prc()
phi_vec_erisir, g_vec_erisir, T_erisir = simulate_erisir_prc()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

axes[0].plot(phi_vec_wb, g_vec_wb, '-k', linewidth=2)
axes[0].plot([0, 1], [1, 0], '--k', linewidth=1)
axes[0].axis([0, 1, 0, 1])
axes[0].set_box_aspect(1)
axes[0].set_xlabel(r'$\varphi$')
axes[0].set_ylabel('$g$')
axes[0].set_title('WB')

axes[1].plot(phi_vec_hh, g_vec_hh, '-k', linewidth=2)
axes[1].axis([0, 1, -0.1, 0.1])
axes[1].set_box_aspect(1)
axes[1].set_xlabel(r'$\varphi$')
axes[1].set_title('HH')

axes[2].plot(phi_vec_erisir, g_vec_erisir, '-k', linewidth=2)
axes[2].axis([0, 1, 0, 0.19])
axes[2].set_box_aspect(1)
axes[2].set_xlabel(r'$\varphi$')
axes[2].set_title('Erisir')

plt.tight_layout()
plt.show()